In [3]:
import os
import numpy as np
import pandas as pd


class InventoryOptimizer:

    def __init__(self, filepath):
        self.filepath = filepath
        os.makedirs("../outputs/inventory", exist_ok=True)

    ############################################################

    def load_data(self):

        print("=" * 60)
        print("Loading Sales Dataset")
        print("=" * 60)

        self.df = pd.read_csv(
            self.filepath,
            parse_dates=["InvoiceDate"],
            dtype={
                "Invoice": str,
                "StockCode": str
            },
            low_memory=False
        )

        print("Dataset Shape:", self.df.shape)

    ############################################################

    def create_daily_product_sales(self):

        print("\nCreating Product Daily Sales...\n")

        self.df["Date"] = self.df["InvoiceDate"].dt.date

        self.daily = (
            self.df
            .groupby(
                ["StockCode", "Description", "Date"],
                as_index=False
            )
            .agg(
                Daily_Quantity=("Quantity", "sum"),
                Daily_Revenue=("Total_Price", "sum")
            )
        )

        print("Daily Product Records:", len(self.daily))

    ############################################################

    def product_statistics(self):

        print("\nCreating Product Statistics...\n")

        self.product = (
            self.daily
            .groupby(
                ["StockCode", "Description"],
                as_index=False
            )
            .agg(
                Revenue=("Daily_Revenue", "sum"),
                Quantity=("Daily_Quantity", "sum"),
                Avg_Daily_Demand=("Daily_Quantity", "mean"),
                Demand_STD=("Daily_Quantity", "std"),
                Max_Demand=("Daily_Quantity", "max"),
                Min_Demand=("Daily_Quantity", "min"),
                Selling_Days=("Date", "count")
            )
        )

        self.product["Demand_STD"] = self.product["Demand_STD"].fillna(0)

        print("Product Statistics Shape:", self.product.shape)

    ############################################################

    def abc_analysis(self):

        print("\nPerforming ABC Analysis...\n")

        # Sort products by revenue
        self.product = (
            self.product
            .sort_values("Revenue", ascending=False)
            .reset_index(drop=True)
        )

        total_revenue = self.product["Revenue"].sum()

        self.product["Revenue_%"] = (
            self.product["Revenue"] / total_revenue
        ) * 100

        self.product["Cumulative_%"] = (
            self.product["Revenue_%"].cumsum()
        )

        def classify(x):
            if x <= 80:
                return "A"
            elif x <= 95:
                return "B"
            else:
                return "C"

        self.product["ABC_Class"] = (
            self.product["Cumulative_%"]
            .apply(classify)
        )

        print("ABC Analysis Summary:")
        print(self.product["ABC_Class"].value_counts())

    ############################################################

    def save(self):

        self.product.to_csv(
            "../outputs/inventory/product_inventory.csv",
            index=False
        )

        print("\nInventory Dataset Saved Successfully")

    ############################################################

    def run(self):

        self.load_data()

        self.create_daily_product_sales()

        self.product_statistics()

        self.abc_analysis()

        self.save()

        print("\nInventory Module Step 1 Completed")


############################################################

if __name__ == "__main__":

    obj = InventoryOptimizer(
        r"C:\Users\ka843\Coding\Amdox Internship_project\data\clean\clean_sales.csv"
    )

    obj.run()

Loading Sales Dataset
Dataset Shape: (1007914, 10)

Creating Product Daily Sales...

Daily Product Records: 534609

Creating Product Statistics...

Product Statistics Shape: (5601, 9)

Performing ABC Analysis...

ABC Analysis Summary:
ABC_Class
C    3023
B    1426
A    1152
Name: count, dtype: int64

Inventory Dataset Saved Successfully

Inventory Module Step 1 Completed


In [ ]:
import os
import numpy as np
import pandas as pd


class InventoryOptimizer:

    def __init__(self, filepath):
        self.filepath = filepath
        os.makedirs("../outputs/inventory", exist_ok=True)

    ############################################################

    def load_data(self):

        print("=" * 60)
        print("Loading Sales Dataset")
        print("=" * 60)

        self.df = pd.read_csv(
            self.filepath,
            parse_dates=["InvoiceDate"],
            dtype={
                "Invoice": str,
                "StockCode": str
            },
            low_memory=False
        )

        print("Dataset Shape:", self.df.shape)

    ############################################################

    def create_daily_product_sales(self):

        print("\nCreating Product Daily Sales...\n")

        self.df["Date"] = self.df["InvoiceDate"].dt.date

        self.daily = (
            self.df
            .groupby(
                ["StockCode", "Description", "Date"],
                as_index=False
            )
            .agg(
                Daily_Quantity=("Quantity", "sum"),
                Daily_Revenue=("Total_Price", "sum")
            )
        )

        print("Daily Product Records:", len(self.daily))

    ############################################################

    def product_statistics(self):

        print("\nCreating Product Statistics...\n")

        self.product = (
            self.daily
            .groupby(
                ["StockCode", "Description"],
                as_index=False
            )
            .agg(
                Revenue=("Daily_Revenue", "sum"),
                Quantity=("Daily_Quantity", "sum"),
                Avg_Daily_Demand=("Daily_Quantity", "mean"),
                Demand_STD=("Daily_Quantity", "std"),
                Max_Demand=("Daily_Quantity", "max"),
                Min_Demand=("Daily_Quantity", "min"),
                Selling_Days=("Date", "count")
            )
        )

        self.product["Demand_STD"] = self.product["Demand_STD"].fillna(0)

        print("Product Statistics Shape:", self.product.shape)

    ############################################################

    def abc_analysis(self):

        print("\nPerforming ABC Analysis...\n")

        self.product = (
            self.product
            .sort_values("Revenue", ascending=False)
            .reset_index(drop=True)
        )

        total_revenue = self.product["Revenue"].sum()

        self.product["Revenue_%"] = (
            self.product["Revenue"] / total_revenue
        ) * 100

        self.product["Cumulative_%"] = (
            self.product["Revenue_%"].cumsum()
        )

        def classify(x):

            if x <= 80:
                return "A"

            elif x <= 95:
                return "B"

            else:
                return "C"

        self.product["ABC_Class"] = (
            self.product["Cumulative_%"]
            .apply(classify)
        )

        print("ABC Analysis Summary:\n")
        print(self.product["ABC_Class"].value_counts())

    ############################################################

    def xyz_analysis(self):

        print("\nPerforming XYZ Analysis...\n")

        self.product["CV"] = np.where(
            self.product["Avg_Daily_Demand"] == 0,
            0,
            self.product["Demand_STD"] /
            self.product["Avg_Daily_Demand"]
        )

        def classify(cv):

            if cv <= 0.5:
                return "X"

            elif cv <= 1:
                return "Y"

            else:
                return "Z"

        self.product["XYZ_Class"] = (
            self.product["CV"]
            .apply(classify)
        )

        print("XYZ Analysis Summary:\n")

        print(
            self.product["XYZ_Class"]
            .value_counts()
        )

    ############################################################

    def inventory_class(self):

        print("\nCreating Inventory Classes...\n")

        self.product["Inventory_Class"] = (

            self.product["ABC_Class"]

            +

            self.product["XYZ_Class"]

        )

        print(

            self.product["Inventory_Class"]

            .value_counts()

        )

    ############################################################

    def save(self):

        self.product.to_csv(
            "../outputs/inventory/product_inventory.csv",
            index=False
        )

        print("\nInventory Dataset Saved Successfully")

    ############################################################

    def run(self):

        self.load_data()

        self.create_daily_product_sales()

        self.product_statistics()

        self.abc_analysis()

        self.xyz_analysis()

        self.inventory_class()

        self.save()

        print("\nInventory Module Step 1 Completed")


############################################################

if __name__ == "__main__":

    obj = InventoryOptimizer(
        r"C:\Users\ka843\Coding\Amdox Internship_project\data\clean\clean_sales.csv"
    )

    obj.run()